In [1]:
!pip install -q torch-geometric

In [2]:
import os
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)

from torch_geometric.nn import RGCNConv

# Project folder path
BASE_DIR = os.getcwd()

# File paths based on your PyCharm project folder
MERGED_DATA_PATH = os.path.join(BASE_DIR, "merged_data.csv")
SAMPLE_DATA_PATH = os.path.join(BASE_DIR, "sample_data.csv")
DRUG_NAMES_PATH = os.path.join(BASE_DIR, "..", "drug_names.tsv")
MEDDRA_INDICATIONS_PATH = os.path.join(BASE_DIR, "meddra_all_indications.tsv")
MEDDRA_SE_PATH = os.path.join(BASE_DIR, "meddra_all_se.tsv")

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Working directory:", BASE_DIR)
print("Using device:", DEVICE)
print("merged_data.csv exists:", os.path.exists(MERGED_DATA_PATH))
print("sample_data.csv exists:", os.path.exists(SAMPLE_DATA_PATH))
print("drug_names.tsv exists:", os.path.exists(DRUG_NAMES_PATH))

/Users/srinadh/Downloads/Adverse Drug Reactions copy/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Working directory: /Users/srinadh/Downloads/Adverse Drug Reactions copy/code files
Using device: cpu
merged_data.csv exists: True
sample_data.csv exists: True
drug_names.tsv exists: True


In [3]:
# -----------------------------
# 2. Reproducibility
# -----------------------------
def set_seed(seed: int = 42):
    import random
    import numpy as np
    import torch

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Ensures reproducibility (important for RGCN training)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

In [4]:
# ============================================
# 3. Config
# ============================================

# Base folder = your current PyCharm notebook/script location
BASE_DIR = os.getcwd()

# Input file
DATA_PATH = os.path.join(BASE_DIR, "merged_data.csv")

# Output folder
SAVE_DIR = os.path.join(BASE_DIR, "rgcn_outputs")
os.makedirs(SAVE_DIR, exist_ok=True)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)
print("DATA_PATH:", DATA_PATH)
print("SAVE_DIR:", SAVE_DIR)

# Data controls
MAX_ROWS = 500_000   # use smaller sample if needed for speed
USE_ONLY_PT = True
MIN_SE_FREQ = 20
NEG_PER_POS = 3

# Model
EMBED_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT = 0.3
MLP_HIDDEN = 256

# Training
LR = 5e-4
WEIGHT_DECAY = 1e-5
EPOCHS = 50
PATIENCE = 5
TRAIN_BATCH_SIZE = 4096
EVAL_BATCH_SIZE = 8192

Using device: cpu
DATA_PATH: /Users/srinadh/Downloads/Adverse Drug Reactions copy/code files/merged_data.csv
SAVE_DIR: /Users/srinadh/Downloads/Adverse Drug Reactions copy/code files/rgcn_outputs


In [5]:
# =========================================================
# 4. Load raw data
# =========================================================

df_original = pd.read_csv(DATA_PATH)

print("Original shape:", df_original.shape)

# Show first few rows
print("\nFirst 5 rows:")
print(df_original.head())

Original shape: (9312040, 8)

First 5 rows:
  stitch_id_flat umls_id_se concept_type  side_effect_name meddra_type  \
0   CID100000085   C0000729          LLT  Abdominal cramps         LLT   
1   CID100000085   C0000729          LLT  Abdominal cramps          PT   
2   CID100000085   C0000729          LLT  Abdominal cramps         LLT   
3   CID100000085   C0000729          LLT  Abdominal cramps          PT   
4   CID100000085   C0000729          LLT  Abdominal cramps         LLT   

   umls_id2        indication_name  drug_name  
0  C0015544      Failure to thrive  carnitine  
1  C0015544      Failure to thrive  carnitine  
2  C0020615          Hypoglycaemia  carnitine  
3  C0020615          Hypoglycaemia  carnitine  
4  C0022661  Renal failure chronic  carnitine  


In [6]:
# ============================================
# 5. Clean data
# ============================================

df = df_original.copy()

# Check required columns
required_cols = [
    "drug_name",
    "side_effect_name",
    "indication_name",
    "meddra_type",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Lowercase + clean text
for col in required_cols:
    df[col] = df[col].fillna("").astype(str).str.strip().str.lower()

# Keep only PT records
if USE_ONLY_PT:
    df = df[df["meddra_type"] == "pt"].copy()

# Remove blank rows
df = df[
    (df["drug_name"] != "") &
    (df["side_effect_name"] != "") &
    (df["indication_name"] != "")
].copy()

print(f"Rows after basic cleaning: {len(df):,}")

# Remove duplicates
df = df.drop_duplicates(
    subset=["drug_name", "indication_name", "side_effect_name"]
).reset_index(drop=True)

print(f"Rows after deduplication: {len(df):,}")

# Remove rare side effects
se_counts = df["side_effect_name"].value_counts()
keep_se = se_counts[se_counts >= MIN_SE_FREQ].index
df = df[df["side_effect_name"].isin(keep_se)].copy().reset_index(drop=True)

print(f"Rows after dropping rare side effects: {len(df):,}")
print(f"Remaining unique side effects: {df['side_effect_name'].nunique():,}")

# Sample down for performance (IMPORTANT for laptop)
if len(df) > MAX_ROWS:
    df = df.sample(n=MAX_ROWS, random_state=42).reset_index(drop=True)

print(f"Rows used for training: {len(df):,}")

# Save cleaned data
output_path = os.path.join(SAVE_DIR, "rgcn_sample_used.csv")
df.to_csv(output_path, index=False)

print(f"Saved cleaned data to: {output_path}")

Rows after basic cleaning: 4,710,469
Rows after deduplication: 2,505,988
Rows after dropping rare side effects: 2,491,362
Remaining unique side effects: 4,394
Rows used for training: 500,000
Saved cleaned data to: /Users/srinadh/Downloads/Adverse Drug Reactions copy/code files/rgcn_outputs/rgcn_sample_used.csv


In [7]:
# =========================================================
# 6. Build vocabularies
# =========================================================

# Unique values
drug_names = sorted(df["drug_name"].unique())
indication_names = sorted(df["indication_name"].unique())
side_effect_names = sorted(df["side_effect_name"].unique())

# Text -> local ID mappings
drug2id = {name: i for i, name in enumerate(drug_names)}
ind2id = {name: i for i, name in enumerate(indication_names)}
se2id = {name: i for i, name in enumerate(side_effect_names)}

# Local ID -> text mappings
id2drug = {i: name for name, i in drug2id.items()}
id2ind = {i: name for name, i in ind2id.items()}
id2se = {i: name for name, i in se2id.items()}

# Counts
num_drugs = len(drug2id)
num_inds = len(ind2id)
num_ses = len(se2id)

# Global node ID offsets
drug_offset = 0
ind_offset = num_drugs
se_offset = num_drugs + num_inds
num_nodes = num_drugs + num_inds + num_ses

# Local -> global ID conversion helpers
def drug_global_id_from_local(drug_id):
    return drug_offset + int(drug_id)

def ind_global_id_from_local(ind_id):
    return ind_offset + int(ind_id)

def se_global_id_from_local(se_id):
    return se_offset + int(se_id)

print("num_drugs :", num_drugs)
print("num_inds  :", num_inds)
print("num_ses   :", num_ses)
print("num_nodes :", num_nodes)

# Optional sanity checks
print("\nSample mappings:")
print("drug sample:", list(drug2id.items())[:3])
print("indication sample:", list(ind2id.items())[:3])
print("side effect sample:", list(se2id.items())[:3])

num_drugs : 1276
num_inds  : 2201
num_ses   : 4392
num_nodes : 7869

Sample mappings:
drug sample: [('1,25(oh)2d3', 0), ('18f-fdg', 1), ('2-hydroxysuccinaldehyde', 2)]
indication sample: [('abdominal abscess', 0), ('abdominal discomfort', 1), ('abdominal distension', 2)]
side effect sample: [('abasia', 0), ('abdominal abscess', 1), ('abdominal aortic bruit', 2)]


In [8]:
# =========================================================
# 7. Convert triples to integer IDs
# =========================================================

# Keep required columns
triples_df = df[["drug_name", "indication_name", "side_effect_name"]].copy()

# Map to IDs
triples_df["drug_id"] = triples_df["drug_name"].map(drug2id)
triples_df["ind_id"] = triples_df["indication_name"].map(ind2id)
triples_df["se_id"] = triples_df["side_effect_name"].map(se2id)

# 🔴 Safety check (VERY IMPORTANT)
if triples_df[["drug_id", "ind_id", "se_id"]].isnull().any().any():
    raise ValueError("Mapping failed: Some IDs are missing (NaN values found)")

# Convert to list of tuples
triples_id = list(
    triples_df[["drug_id", "ind_id", "se_id"]]
    .astype(int)
    .itertuples(index=False, name=None)
)

# Store all positives
all_positive_set = set(triples_id)

# Split into train/test
train_pos, test_pos = train_test_split(
    triples_id, test_size=0.15, random_state=42
)

# Split train into train/val
train_pos, val_pos = train_test_split(
    train_pos, test_size=0.15, random_state=42
)

print(f"Train positives: {len(train_pos):,}")
print(f"Val positives:   {len(val_pos):,}")
print(f"Test positives:  {len(test_pos):,}")

# Optional sanity check
print("\nSample triple:", train_pos[0])

Train positives: 361,250
Val positives:   63,750
Test positives:  75,000

Sample triple: (110, 502, 296)


In [9]:
# =========================================================
# 8. Build graph edges for message passing
# =========================================================

# Relation types
REL_HAS_IND = 0         # drug -> indication
REL_CAUSES_SE = 1       # drug -> side effect
REL_REV_HAS_IND = 2     # indication -> drug
REL_REV_CAUSES_SE = 3   # side effect -> drug
NUM_RELATIONS = 4

# Edge storage
edge_src = []
edge_dst = []
edge_type_list = []

# Build graph only from training positives
for d_id, i_id, s_id in train_pos:
    d_global = drug_global_id_from_local(d_id)
    i_global = ind_global_id_from_local(i_id)
    s_global = se_global_id_from_local(s_id)

    # drug -> indication
    edge_src.append(d_global)
    edge_dst.append(i_global)
    edge_type_list.append(REL_HAS_IND)

    # drug -> side effect
    edge_src.append(d_global)
    edge_dst.append(s_global)
    edge_type_list.append(REL_CAUSES_SE)

    # indication -> drug
    edge_src.append(i_global)
    edge_dst.append(d_global)
    edge_type_list.append(REL_REV_HAS_IND)

    # side effect -> drug
    edge_src.append(s_global)
    edge_dst.append(d_global)
    edge_type_list.append(REL_REV_CAUSES_SE)

# Convert to tensors
edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long, device=DEVICE)
edge_type = torch.tensor(edge_type_list, dtype=torch.long, device=DEVICE)

print("edge_index shape:", edge_index.shape)
print("edge_type shape :", edge_type.shape)
print("num_edges       :", edge_index.shape[1])

# Optional quick sanity check
print("First 5 edges:")
for idx in range(min(5, edge_index.shape[1])):
    print(
        f"src={edge_index[0, idx].item()}, "
        f"dst={edge_index[1, idx].item()}, "
        f"rel={edge_type[idx].item()}"
    )

edge_index shape: torch.Size([2, 1445000])
edge_type shape : torch.Size([1445000])
num_edges       : 1445000
First 5 edges:
src=110, dst=1778, rel=0
src=110, dst=3773, rel=1
src=1778, dst=110, rel=2
src=3773, dst=110, rel=3
src=730, dst=3079, rel=0


In [10]:
import os
import torch

SAVE_DIR = "/Users/srinadh/Downloads/Adverse Drug Reactions copy/code files"

os.makedirs(SAVE_DIR, exist_ok=True)

torch.save(edge_index, os.path.join(SAVE_DIR, "edge_index.pt"))
torch.save(edge_type, os.path.join(SAVE_DIR, "edge_type.pt"))

print("✅ Graph saved successfully")

✅ Graph saved successfully


In [11]:
# =========================================================
# 9. Negative sampling
# =========================================================

def sample_negative_triples(pos_triples, neg_per_pos=3):
    negatives = []
    used = set()

    drug_ids = np.arange(num_drugs)
    ind_ids = np.arange(num_inds)
    se_ids = np.arange(num_ses)

    target_neg = int(len(pos_triples) * neg_per_pos)

    # Safety counter to avoid infinite loop
    max_attempts = target_neg * 10
    attempts = 0

    while len(negatives) < target_neg and attempts < max_attempts:
        attempts += 1

        d_id, i_id, s_id = random.choice(pos_triples)

        corrupt_which = random.choices(
            ["drug", "ind", "se"],
            weights=[0.2, 0.2, 0.6],
            k=1
        )[0]

        if corrupt_which == "drug":
            d_new = int(np.random.choice(drug_ids))
            candidate = (d_new, i_id, s_id)

        elif corrupt_which == "ind":
            i_new = int(np.random.choice(ind_ids))
            candidate = (d_id, i_new, s_id)

        else:
            s_new = int(np.random.choice(se_ids))
            candidate = (d_id, i_id, s_new)

        # Skip if real or already used
        if candidate in all_positive_set or candidate in used:
            continue

        used.add(candidate)
        negatives.append(candidate)

    if len(negatives) < target_neg:
        print(f"Warning: Only generated {len(negatives)} negatives out of {target_neg}")

    return negatives


# Generate negatives
train_neg = sample_negative_triples(train_pos, NEG_PER_POS)
val_neg = sample_negative_triples(val_pos, NEG_PER_POS)
test_neg = sample_negative_triples(test_pos, NEG_PER_POS)

print(f"Train negatives: {len(train_neg):,}")
print(f"Val negatives:   {len(val_neg):,}")
print(f"Test negatives:  {len(test_neg):,}")

# Optional sanity check
print("Sample negative:", train_neg[0])

Train negatives: 1,083,750
Val negatives:   191,250
Test negatives:  225,000
Sample negative: (1126, 1995, 2403)


In [12]:
# =========================================================
# 10. Build datasets
# =========================================================

def build_examples(pos_triples, neg_triples):
    # Combine positives and negatives
    triples_out = pos_triples + neg_triples

    # Labels: 1 = positive, 0 = negative
    labels = np.array(
        [1.0] * len(pos_triples) + [0.0] * len(neg_triples),
        dtype=np.float32
    )

    # Shuffle indices
    idx = np.arange(len(triples_out))
    np.random.shuffle(idx)

    # Apply shuffle
    triples_out = np.array(triples_out, dtype=np.int64)[idx]
    labels = labels[idx]

    return triples_out, labels


# Build datasets
train_examples, y_train = build_examples(train_pos, train_neg)
val_examples, y_val = build_examples(val_pos, val_neg)
test_examples, y_test = build_examples(test_pos, test_neg)

print("train_examples:", train_examples.shape)
print("val_examples  :", val_examples.shape)
print("test_examples :", test_examples.shape)

# 🔥 Optional sanity check
print("\nSample train example:", train_examples[0])
print("Sample label:", y_train[0])

# 🔥 Check class balance
print("\nTrain positive ratio:", y_train.mean())
print("Val positive ratio  :", y_val.mean())
print("Test positive ratio :", y_test.mean())

train_examples: (1445000, 3)
val_examples  : (255000, 3)
test_examples : (300000, 3)

Sample train example: [ 383 1414 2072]
Sample label: 1.0

Train positive ratio: 0.25
Val positive ratio  : 0.25
Test positive ratio : 0.25


In [13]:
# =========================================================
# 11. Minibatch iterator
# =========================================================
def iterate_minibatches(examples, labels, batch_size=4096, shuffle=True):
    n = len(examples)
    idx = np.arange(n)

    if shuffle:
        np.random.shuffle(idx)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_idx = idx[start:end]

        batch_x = examples[batch_idx]
        batch_y = labels[batch_idx]

        # 🔥 Convert to torch tensors (important for training)
        batch_x = torch.tensor(batch_x, dtype=torch.long, device=DEVICE)
        batch_y = torch.tensor(batch_y, dtype=torch.float32, device=DEVICE)

        yield batch_x, batch_y

In [14]:
# =========================================================
# 12. Model
# =========================================================
class RGCNEncoder(nn.Module):
    def __init__(self, num_nodes, num_relations, emb_dim=64, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()

        self.node_emb = nn.Embedding(num_nodes, emb_dim)
        self.convs = nn.ModuleList()

        if num_layers == 1:
            self.convs.append(RGCNConv(emb_dim, hidden_dim, num_relations))
        else:
            self.convs.append(RGCNConv(emb_dim, hidden_dim, num_relations))
            for _ in range(num_layers - 2):
                self.convs.append(RGCNConv(hidden_dim, hidden_dim, num_relations))
            self.convs.append(RGCNConv(hidden_dim, hidden_dim, num_relations))

        self.dropout = dropout

    def forward(self, edge_index, edge_type):
        x = self.node_emb.weight

        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index, edge_type)

            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)

        return x


class MLPTripleScorer(nn.Module):
    def __init__(self, hidden_dim, mlp_hidden=256, dropout=0.3):
        super().__init__()

        input_dim = hidden_dim * 5   # d, i, s, d*s, i*s

        self.net = nn.Sequential(
            nn.Linear(input_dim, mlp_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden, mlp_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(mlp_hidden // 2, 1)
        )

    def forward(self, z_drug, z_ind, z_se):
        x = torch.cat(
            [
                z_drug,
                z_ind,
                z_se,
                z_drug * z_se,
                z_ind * z_se,
            ],
            dim=-1
        )
        return self.net(x).squeeze(-1)


class ADRRGCNModel(nn.Module):
    def __init__(self, num_nodes, num_relations, emb_dim=64, hidden_dim=128, num_layers=2, dropout=0.3):
        super().__init__()

        self.encoder = RGCNEncoder(
            num_nodes=num_nodes,
            num_relations=num_relations,
            emb_dim=emb_dim,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            dropout=dropout,
        )

        self.scorer = MLPTripleScorer(
            hidden_dim=hidden_dim,
            mlp_hidden=MLP_HIDDEN,
            dropout=dropout,
        )

    def encode(self, edge_index, edge_type):
        return self.encoder(edge_index, edge_type)

    def score_batch(self, z, triples_batch):
        # triples_batch is expected to already be a torch tensor
        if not torch.is_tensor(triples_batch):
            triples_batch = torch.tensor(triples_batch, dtype=torch.long, device=z.device)
        else:
            triples_batch = triples_batch.to(z.device)

        d_local = triples_batch[:, 0]
        i_local = triples_batch[:, 1]
        s_local = triples_batch[:, 2]

        d_global = d_local + drug_offset
        i_global = i_local + ind_offset
        s_global = s_local + se_offset

        z_drug = z[d_global]
        z_ind = z[i_global]
        z_se = z[s_global]

        return self.scorer(z_drug, z_ind, z_se)

In [15]:
# =========================================================
# 13. Evaluation helpers
# =========================================================

@torch.no_grad()
def get_probs(model, edge_index, edge_type, examples, batch_size=8192):
    model.eval()
    z = model.encode(edge_index, edge_type)

    all_probs = []
    dummy_labels = np.zeros(len(examples), dtype=np.float32)

    for batch_examples, _ in iterate_minibatches(
        examples, dummy_labels, batch_size=batch_size, shuffle=False
    ):
        logits = model.score_batch(z, batch_examples)
        probs = torch.sigmoid(logits)
        all_probs.append(probs.detach().cpu().numpy())

    return np.concatenate(all_probs, axis=0)


def find_best_threshold(y_true, probs):
    thresholds = np.arange(0.10, 0.91, 0.05)
    best_thr = 0.5
    best_f1 = -1.0

    for thr in thresholds:
        preds = (probs >= thr).astype(int)
        f1 = f1_score(y_true, preds, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_thr = thr

    return best_thr, best_f1


@torch.no_grad()
def evaluate(model, edge_index, edge_type, examples, labels, batch_size=8192, threshold=0.5):
    model.eval()
    z = model.encode(edge_index, edge_type)

    all_logits = []
    all_probs = []

    for batch_examples, batch_labels in iterate_minibatches(
        examples, labels, batch_size=batch_size, shuffle=False
    ):
        logits = model.score_batch(z, batch_examples)
        probs = torch.sigmoid(logits)

        all_logits.append(logits.detach().cpu().numpy())
        all_probs.append(probs.detach().cpu().numpy())

    logits = np.concatenate(all_logits, axis=0)
    probs = np.concatenate(all_probs, axis=0)
    labels = np.asarray(labels).astype(int)
    preds = (probs >= threshold).astype(int)

    metrics = {}

    # Safe metric calculation
    try:
        metrics["roc_auc"] = roc_auc_score(labels, probs)
    except ValueError:
        metrics["roc_auc"] = 0.0

    try:
        metrics["pr_auc"] = average_precision_score(labels, probs)
    except ValueError:
        metrics["pr_auc"] = 0.0

    metrics["f1"] = f1_score(labels, preds, zero_division=0)
    metrics["precision"] = precision_score(labels, preds, zero_division=0)
    metrics["recall"] = recall_score(labels, preds, zero_division=0)

    return metrics, probs

In [16]:
# =========================================================
# 14. Initialize model / optimizer / loss
# =========================================================

# Create model
model = ADRRGCNModel(
    num_nodes=num_nodes,
    num_relations=NUM_RELATIONS,
    emb_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(DEVICE)

# Move graph tensors to same device
edge_index = edge_index.to(DEVICE)
edge_type = edge_type.to(DEVICE)

# Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

# Class imbalance handling
num_pos = len(train_pos)
num_neg = len(train_neg)
pos_weight_value = num_neg / max(num_pos, 1)

print("num_pos:", num_pos)
print("num_neg:", num_neg)
print("pos_weight:", pos_weight_value)

# Loss function
criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32, device=DEVICE)
)

num_pos: 361250
num_neg: 1083750
pos_weight: 3.0


In [17]:
# =========================================================
# 15. Train (20 EPOCHS + EARLY STOPPING)
# =========================================================

EPOCHS = 20
PATIENCE = 3   # stops if validation PR-AUC does not improve for 3 consecutive epochs

best_val_pr = -np.inf
best_state = None
best_threshold = 0.5
patience_counter = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for batch_examples, batch_labels in iterate_minibatches(
        train_examples, y_train, batch_size=TRAIN_BATCH_SIZE, shuffle=True
    ):
        optimizer.zero_grad()

        z = model.encode(edge_index, edge_type)
        logits = model.score_batch(z, batch_examples)

        loss = criterion(logits, batch_labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_examples.size(0)
        total_examples += batch_examples.size(0)

    train_loss = total_loss / max(total_examples, 1)

    val_probs = get_probs(
        model,
        edge_index,
        edge_type,
        val_examples,
        batch_size=EVAL_BATCH_SIZE
    )

    epoch_best_thr, _ = find_best_threshold(y_val.astype(int), val_probs)

    val_metrics, _ = evaluate(
        model,
        edge_index,
        edge_type,
        val_examples,
        y_val,
        batch_size=EVAL_BATCH_SIZE,
        threshold=epoch_best_thr
    )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "best_threshold": epoch_best_thr,
        **val_metrics
    }
    history.append(row)

    print(
        f"Epoch {epoch:03d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val ROC-AUC: {val_metrics['roc_auc']:.4f} | "
        f"Val PR-AUC: {val_metrics['pr_auc']:.4f} | "
        f"Val F1: {val_metrics['f1']:.4f} | "
        f"Val Precision: {val_metrics['precision']:.4f} | "
        f"Val Recall: {val_metrics['recall']:.4f} | "
        f"Thr: {epoch_best_thr:.2f}"
    )

    if val_metrics["pr_auc"] > best_val_pr:
        best_val_pr = val_metrics["pr_auc"]
        best_threshold = epoch_best_thr
        best_state = {
            k: v.detach().cpu().clone()
            for k, v in model.state_dict().items()
        }
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"No improvement. Early stopping counter: {patience_counter}/{PATIENCE}")

        if patience_counter >= PATIENCE:
            print("Early stopping triggered.")
            break 

Epoch 001 | Train Loss: 0.7731 | Val ROC-AUC: 0.9047 | Val PR-AUC: 0.7209 | Val F1: 0.7106 | Val Precision: 0.6475 | Val Recall: 0.7873 | Thr: 0.80
Epoch 002 | Train Loss: 0.5435 | Val ROC-AUC: 0.9334 | Val PR-AUC: 0.7833 | Val F1: 0.7636 | Val Precision: 0.6829 | Val Recall: 0.8658 | Thr: 0.75
Epoch 003 | Train Loss: 0.4684 | Val ROC-AUC: 0.9435 | Val PR-AUC: 0.8052 | Val F1: 0.7856 | Val Precision: 0.7056 | Val Recall: 0.8860 | Thr: 0.75
Epoch 004 | Train Loss: 0.4275 | Val ROC-AUC: 0.9481 | Val PR-AUC: 0.8160 | Val F1: 0.7976 | Val Precision: 0.7216 | Val Recall: 0.8915 | Thr: 0.80
Epoch 005 | Train Loss: 0.4008 | Val ROC-AUC: 0.9523 | Val PR-AUC: 0.8273 | Val F1: 0.8093 | Val Precision: 0.7431 | Val Recall: 0.8884 | Thr: 0.80
Epoch 006 | Train Loss: 0.3802 | Val ROC-AUC: 0.9553 | Val PR-AUC: 0.8348 | Val F1: 0.8175 | Val Precision: 0.7548 | Val Recall: 0.8917 | Thr: 0.80
Epoch 007 | Train Loss: 0.3650 | Val ROC-AUC: 0.9581 | Val PR-AUC: 0.8428 | Val F1: 0.8250 | Val Precision: 0.75

KeyboardInterrupt: 

In [18]:
import os
import torch
import pandas as pd

# Load best weights into model before saving
if best_state is not None:
    model.load_state_dict(best_state)

# Save paths
model_path = os.path.join(SAVE_DIR, "best_rgcn_model.pt")
history_path = os.path.join(SAVE_DIR, "training_history.csv")

# Save model checkpoint
checkpoint = {
    "model_state_dict": model.state_dict(),
    "best_val_pr": best_val_pr,
    "best_threshold": best_threshold,
    "num_nodes": num_nodes,
    "num_relations": NUM_RELATIONS,
    "emb_dim": EMBED_DIM,
    "hidden_dim": HIDDEN_DIM,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "mlp_hidden": MLP_HIDDEN,
    "drug2id": drug2id,
    "ind2id": ind2id,
    "se2id": se2id,
    "id2drug": id2drug,
    "id2ind": id2ind,
    "id2se": id2se,
}

torch.save(checkpoint, model_path)

# Save training history
pd.DataFrame(history).to_csv(history_path, index=False)

print("Model saved to:", model_path)
print("History saved to:", history_path)
print("Best validation PR-AUC:", best_val_pr)
print("Best threshold:", best_threshold)

Model saved to: /Users/srinadh/Downloads/Adverse Drug Reactions copy/code files/best_rgcn_model.pt
History saved to: /Users/srinadh/Downloads/Adverse Drug Reactions copy/code files/training_history.csv
Best validation PR-AUC: 0.8514121577486721
Best threshold: 0.8000000000000003


In [19]:
# =========================================================
# 16. Restore best model
# =========================================================
if best_state is not None:
    model.load_state_dict(best_state)

print("Best validation PR-AUC:", best_val_pr)
print("Best validation threshold:", best_threshold)


Best validation PR-AUC: 0.8514121577486721
Best validation threshold: 0.8000000000000003


In [20]:
# =========================================================
# 17. Save training outputs
# =========================================================
torch.save(model.state_dict(), os.path.join(SAVE_DIR, "rgcn_model_v2.pth"))
pd.DataFrame(history).to_csv(
    os.path.join(SAVE_DIR, "training_history_v2.csv"),
    index=False
)

In [21]:
# =========================================================
# 18. Final test
# =========================================================
test_metrics, test_probs = evaluate(
    model,
    edge_index,
    edge_type,
    test_examples,
    y_test,
    batch_size=EVAL_BATCH_SIZE,
    threshold=best_threshold
)

print("\n===== Test Metrics =====")
print(f"Best threshold from val: {best_threshold:.2f}")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

pd.DataFrame([{
    "best_threshold": best_threshold,
    **test_metrics
}]).to_csv(
    os.path.join(SAVE_DIR, "test_metrics_v2.csv"),
    index=False
)



===== Test Metrics =====
Best threshold from val: 0.80
roc_auc: 0.9619
pr_auc: 0.8528
f1: 0.8371
precision: 0.7800
recall: 0.9031


In [22]:
import os
import pandas as pd
import torch

BASE_DIR = os.getcwd()

DATA_PATH = os.path.join(BASE_DIR, "sample_data.csv")
SAVE_DIR = os.path.join(BASE_DIR, "rgcn_outputs")

os.makedirs(SAVE_DIR, exist_ok=True)

print("Working directory:", BASE_DIR)
print("Data exists:", os.path.exists(DATA_PATH))
print("Save folder exists:", os.path.exists(SAVE_DIR))
print("Device:", torch.device("cuda" if torch.cuda.is_available() else "cpu"))

df = pd.read_csv(DATA_PATH)
print("Data shape:", df.shape)
print(df.head())

Working directory: /Users/srinadh/Downloads/Adverse Drug Reactions copy/code files
Data exists: True
Save folder exists: True
Device: cpu
Data shape: (125369, 8)
  stitch_id_flat umls_id_se concept_type             side_effect_name  \
0   CID100000085   C0003811           PT                   Arrhythmia   
1   CID100000085   C0023218          LLT  Cramps of lower extremities   
2   CID100000085   C0013395           PT                    Dyspepsia   
3   CID100000085   C0151735          LLT      Injection site reaction   
4   CID100000085   C0027497          LLT                       Nausea   

  meddra_type  umls_id2       indication_name  drug_name  
0          PT  C0878544        Cardiomyopathy  carnitine  
1          PT  C1142132  Carnitine deficiency  carnitine  
2          PT  C0026827             Hypotonia  carnitine  
3         LLT  C0878544        Cardiomyopathy  carnitine  
4         LLT  C0020615         Hypoglycaemia  carnitine  
